# LoRA & QLoRA: Parameter-Efficient Fine-Tuning

This notebook explores **LoRA** (Low-Rank Adaptation) and **QLoRA** (Quantized LoRA), techniques
that make fine-tuning large language models practical by updating only a tiny fraction of parameters.

We will:
1. Explain why full fine-tuning is impractical for billion-parameter models
2. Derive the LoRA math: low-rank weight updates $\Delta W = BA$
3. Implement LoRA from scratch in pure PyTorch
4. Use the PEFT library for production LoRA
5. Understand QLoRA: 4-bit quantization + LoRA for extreme memory savings
6. Fine-tune LLaMA 3.2 1B on an instruction dataset with QLoRA

**References:**
- Hu et al. (2021). *LoRA: Low-Rank Adaptation of Large Language Models.* https://arxiv.org/abs/2106.09685
- Dettmers et al. (2023). *QLoRA: Efficient Finetuning of Quantized LLMs.* https://arxiv.org/abs/2305.14314

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import torch
import torch.nn as nn
import math
import numpy as np
import pandas as pd
from src.utils.device import set_device, set_seed

device = set_device()
set_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

## 1. The Fine-Tuning Problem

Full fine-tuning updates **every parameter** in the model. For a 1B parameter model:

| Component | Memory (fp32) | Memory (fp16) |
|---|---|---|
| Model parameters | 4 GB | 2 GB |
| Gradients | 4 GB | 2 GB |
| Optimizer states (Adam: 2x) | 8 GB | 8 GB (kept in fp32) |
| **Total** | **16 GB** | **12 GB** |

And that's *before* activations. A single A100 (80 GB) can barely fine-tune a 7B model.

**LoRA's insight:** The weight updates during fine-tuning have low intrinsic rank.
We don't need to update all $d \times d$ parameters — a rank-$r$ update (where $r \ll d$)
captures most of the adaptation, using only ~0.1-1% of the parameters.

## 2. LoRA: Low-Rank Adaptation

Instead of updating the full weight matrix $W \in \mathbb{R}^{d \times d}$, LoRA decomposes
the update into two low-rank matrices:

$W' = W + \Delta W = W + BA$

where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times d}$, with rank $r \ll d$.

**Parameter count:** $2dr$ instead of $d^2$. For $d = 2048$, $r = 8$: $32{,}768$ vs $4{,}194{,}304$ — a **128x reduction**.

**Scaling factor:** The output is scaled by $\alpha / r$ to control the magnitude of the update:

$h = Wx + \frac{\alpha}{r} BAx$

**Initialization:**
- $B$ is initialized to **zeros**
- $A$ is initialized with random values (Kaiming uniform)
- This ensures $\Delta W = BA = 0$ at the start — the model begins identical to the pretrained version

## 3. Which Layers to Adapt?

The original LoRA paper tested different combinations on GPT-3:

| Target modules | Quality | Parameters |
|---|---|---|
| $W_q$ only | Good | Fewest |
| $W_q + W_v$ | **Best** | Moderate |
| $W_q + W_k + W_v + W_o$ | Good | More |
| All linear layers | Good | Most |

**Modern practice:** Apply LoRA to all linear layers in the attention block ($q$, $k$, $v$, $o$ projections)
and sometimes the MLP layers too. The rank $r$ can be kept small (4-64) since each layer
gets its own adapter.

In [ ]:
# Parameter savings for different configurations
d = 2048  # LLaMA 3.2 1B hidden dimension
n_layers = 16  # number of transformer layers
n_linear_per_layer = 7  # q, k, v, o, gate, up, down projections

full_params = n_layers * n_linear_per_layer * d * d
print(f"Full fine-tuning: {full_params:,} updatable parameters")
print()

for r in [4, 8, 16, 32, 64]:
    lora_params = n_layers * n_linear_per_layer * 2 * d * r
    pct = 100 * lora_params / full_params
    print(f"LoRA rank={r:2d}: {lora_params:>10,} params ({pct:.2f}% of full)")

## 4. LoRA From Scratch

In [ ]:
class LoRALinear(nn.Module):
    """
    Linear layer with LoRA: output = original(x) + (x @ A^T @ B^T) * (alpha / r)
    
    Original weights are frozen. Only A and B are trainable.
    B=zeros, A=random at init, so ΔW = BA = 0 (model starts unchanged).
    """
    def __init__(self, original_linear, r=8, alpha=16):
        super().__init__()
        self.original = original_linear
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        in_features = original_linear.in_features
        out_features = original_linear.out_features

        # Freeze original weights
        for param in self.original.parameters():
            param.requires_grad = False

        # LoRA matrices: B is out×r, A is r×in
        self.lora_A = nn.Parameter(torch.empty(r, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x):
        original_out = self.original(x)
        lora_out = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return original_out + lora_out


# Demo
linear = nn.Linear(64, 64)
lora_linear = LoRALinear(linear, r=4, alpha=8)

total = sum(p.numel() for p in lora_linear.parameters())
trainable = sum(p.numel() for p in lora_linear.parameters() if p.requires_grad)
print(f"Total params: {total:,}")
print(f"Trainable (LoRA only): {trainable:,} ({100*trainable/total:.1f}%)")
print(f"Original frozen: {total - trainable:,}")

In [ ]:
class LoRAModel(nn.Module):
    """Wrap any model, replacing target Linear layers with LoRA variants."""

    def __init__(self, model, target_modules, r=8, alpha=16):
        super().__init__()
        self.model = model

        # Freeze all base parameters
        for param in self.model.parameters():
            param.requires_grad = False

        # Replace matching layers
        self._replaced = 0
        self._apply_lora(self.model, target_modules, r, alpha)
        print(f"Replaced {self._replaced} layers with LoRA (r={r}, alpha={alpha})")

    def _apply_lora(self, module, targets, r, alpha):
        for name, child in module.named_children():
            if isinstance(child, nn.Linear) and any(t in name for t in targets):
                setattr(module, name, LoRALinear(child, r=r, alpha=alpha))
                self._replaced += 1
            else:
                self._apply_lora(child, targets, r, alpha)

    def forward(self, *args, **kwargs):
        return self.model(*args, **kwargs)

    def print_trainable_parameters(self):
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


# Demo: apply to a small transformer-like model
class TinyTransformerBlock(nn.Module):
    def __init__(self, d=128):
        super().__init__()
        self.q_proj = nn.Linear(d, d)
        self.k_proj = nn.Linear(d, d)
        self.v_proj = nn.Linear(d, d)
        self.o_proj = nn.Linear(d, d)
        self.mlp_up = nn.Linear(d, d * 4)
        self.mlp_down = nn.Linear(d * 4, d)

    def forward(self, x):
        q, k, v = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        attn = torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1]), dim=-1) @ v
        x = x + self.o_proj(attn)
        x = x + self.mlp_down(torch.relu(self.mlp_up(x)))
        return x


block = TinyTransformerBlock(d=128)
lora_block = LoRAModel(block, target_modules=["q_proj", "v_proj"], r=8, alpha=16)
lora_block.print_trainable_parameters()

In [ ]:
# Verify: output matches original at initialization (B=0 → ΔW=0)
block_orig = TinyTransformerBlock(d=128)

# Copy weights to a LoRA version
import copy
block_copy = copy.deepcopy(block_orig)
lora_test = LoRAModel(block_copy, target_modules=["q_proj", "v_proj"], r=8, alpha=16)

x = torch.randn(2, 8, 128)
out_orig = block_orig(x)
out_lora = lora_test(x)

max_diff = (out_orig - out_lora).abs().max().item()
print(f"Max output difference at init: {max_diff:.2e}")
print(f"Outputs match: {torch.allclose(out_orig, out_lora, atol=1e-6)}")
print("(Expected: identical, since B=0 means ΔW=0)")

## 5. PEFT: The Practical Way

In practice, use the [PEFT](https://github.com/huggingface/peft) library instead of from-scratch LoRA.
PEFT provides:

- `LoraConfig` — configure rank, alpha, target modules, dropout
- `get_peft_model()` — apply LoRA to any HuggingFace model
- `print_trainable_parameters()` — verify parameter counts
- **Adapter merging** — merge LoRA weights back into the base model for inference
- **Save/load adapters** — save only the tiny LoRA weights (MBs, not GBs)
- **Quantization integration** — seamless QLoRA with bitsandbytes

```bash
pip install -e ".[llm]"  # installs peft, bitsandbytes, trl, datasets, accelerate
```

In [ ]:
# PEFT demo with a small model
try:
    from peft import LoraConfig, get_peft_model, TaskType
    from transformers import AutoModelForCausalLM, AutoTokenizer

    # Use GPT-2 small for a quick local demo
    model_name = "gpt2"
    model = AutoModelForCausalLM.from_pretrained(model_name)

    peft_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["c_attn", "c_proj"],  # GPT-2 attention layers
    )

    peft_model = get_peft_model(model, peft_config)
    peft_model.print_trainable_parameters()

except ImportError:
    print("PEFT not installed. Run: pip install -e '.[llm]'")
    print("Skipping PEFT demo.")

## 6. Adapter Operations

LoRA adapters are independent of the base model:

- **Save adapter only:** `model.save_pretrained("my-adapter/")` — saves ~10-50 MB instead of GBs
- **Load adapter:** `PeftModel.from_pretrained(base_model, "my-adapter/")`
- **Merge into base:** `model.merge_and_unload()` — folds $BA$ into $W$ permanently, removing overhead
- **Stack adapters:** Load multiple task-specific adapters on the same base model

This means you can share a single 1B base model and swap tiny adapters for different tasks.

## 7. QLoRA: Quantized LoRA

QLoRA (Dettmers et al., 2023) pushes memory savings further with three innovations:

**1. 4-bit NormalFloat (NF4) quantization** — quantizes the base model to 4 bits using a
data type optimized for normally distributed weights. Memory: $d^2 / 2$ bytes vs $4d^2$.

**2. Double quantization** — quantizes the quantization constants themselves, saving an
additional ~0.5 bits per parameter.

**3. Paged optimizers** — uses CPU memory for optimizer states when GPU runs out,
paging them back during the update step.

| Method | 1B Model Memory | Trainable Params |
|---|---|---|
| Full fine-tuning (fp32) | ~16 GB | 1B (100%) |
| Full fine-tuning (fp16) | ~12 GB | 1B (100%) |
| LoRA (fp16 base) | ~2.5 GB | ~5M (0.5%) |
| **QLoRA (4-bit base)** | **~1 GB** | **~5M (0.5%)** |

## 8. BitsAndBytesConfig

The `BitsAndBytesConfig` from HuggingFace transformers controls quantization:

```python
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                   # Quantize to 4 bits
    bnb_4bit_quant_type="nf4",           # NormalFloat4 (optimal for pretrained weights)
    bnb_4bit_compute_dtype=torch.float16, # Compute in fp16 during forward
    bnb_4bit_use_double_quant=True,      # Quantize the quantization constants too
)
```

The base model is frozen in 4-bit. LoRA adapters ($A$ and $B$ matrices) train in fp16.
During forward pass, 4-bit weights are dequantized to fp16 on-the-fly.

In [ ]:
# Memory comparison for LLaMA 3.2 1B
params_1b = 1_000_000_000  # ~1B parameters

methods = {
    "Full FT (fp32)": {"model": 4, "grads": 4, "optim": 8, "trainable_pct": 100},
    "Full FT (fp16)": {"model": 2, "grads": 2, "optim": 8, "trainable_pct": 100},
    "LoRA (fp16)": {"model": 2, "grads": 0.01, "optim": 0.04, "trainable_pct": 0.5},
    "QLoRA (4-bit)": {"model": 0.5, "grads": 0.005, "optim": 0.02, "trainable_pct": 0.5},
}

rows = []
for name, m in methods.items():
    total_gb = params_1b * (m["model"] + m["grads"] + m["optim"]) / 1e9
    rows.append({
        "Method": name,
        "Model (GB)": f"{params_1b * m['model'] / 1e9:.1f}",
        "Grads (GB)": f"{params_1b * m['grads'] / 1e9:.2f}",
        "Optim (GB)": f"{params_1b * m['optim'] / 1e9:.2f}",
        "Total (GB)": f"{total_gb:.1f}",
        "Trainable": f"{m['trainable_pct']}%",
    })

print(pd.DataFrame(rows).to_string(index=False))

## 9. Fine-Tuning LLaMA 3.2 1B with QLoRA

We'll fine-tune [meta-llama/Llama-3.2-1B](https://huggingface.co/meta-llama/Llama-3.2-1B)
on a small instruction-following dataset using QLoRA + SFTTrainer from TRL.

**Requirements:**
- CUDA GPU (use the Modal section below if running locally on Mac)
- HuggingFace account with Meta LLaMA license accepted
- `pip install -e ".[llm]"`

**Note:** If you don't have CUDA locally, skip to Section 11 (Run on GPU with Modal).

In [ ]:
# This cell requires CUDA + LLM dependencies
if device != "cuda":
    print(f"Current device: {device}")
    print("Fine-tuning LLaMA requires CUDA. Skip to Section 11 to run via Modal.")
else:
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        BitsAndBytesConfig,
        TrainingArguments,
    )
    from peft import LoraConfig, get_peft_model
    from trl import SFTTrainer
    from datasets import load_dataset

    MODEL_NAME = "meta-llama/Llama-3.2-1B"

    # 4-bit quantization config
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    # Load model in 4-bit
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model.config.use_cache = False

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token

    # LoRA config
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
    )

    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
    print(f"\nModel loaded on: {next(model.parameters()).device}")

In [ ]:
# Load a small instruction dataset
if device == "cuda":
    dataset = load_dataset("mlabonne/guanaco-llama2-1k", split="train")
    print(f"Dataset size: {len(dataset)} samples")
    print(f"Sample:\n{dataset[0]['text'][:300]}...")
else:
    print("Skipping dataset load (no CUDA). Use Modal section below.")

In [ ]:
# Configure and run training
if device == "cuda":
    training_args = TrainingArguments(
        output_dir="./lora-llama3.2-1b",
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        weight_decay=0.01,
        warmup_ratio=0.03,
        logging_steps=10,
        save_strategy="epoch",
        fp16=True,
        optim="paged_adamw_32bit",
        gradient_checkpointing=True,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        args=training_args,
        tokenizer=tokenizer,
        max_seq_length=512,
    )

    print("Starting training...")
    trainer.train()
    print("Training complete!")
else:
    print("Skipping training (no CUDA). Use Modal section below.")

In [ ]:
# Inference: compare base vs fine-tuned
if device == "cuda":
    prompt = "What are the main benefits of using LoRA for fine-tuning large language models?"

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Prompt: {prompt}")
    print(f"\nResponse:\n{response[len(prompt):]}")
else:
    print("Skipping inference (no CUDA). Use Modal section below.")

In [ ]:
# Save and load adapter
if device == "cuda":
    adapter_path = "./lora-llama3.2-1b/adapter"
    model.save_pretrained(adapter_path)
    print(f"Adapter saved to {adapter_path}")

    # Show how small the adapter is
    import os
    adapter_size = sum(
        os.path.getsize(os.path.join(adapter_path, f))
        for f in os.listdir(adapter_path)
        if os.path.isfile(os.path.join(adapter_path, f))
    )
    print(f"Adapter size: {adapter_size / 1024**2:.1f} MB")
    print(f"(vs ~2 GB for the full model in fp16)")
else:
    print("Skipping adapter save (no CUDA).")

## 10. Run on GPU (Modal)

If you don't have a local CUDA GPU, use [Modal](https://modal.com) to run the fine-tuning
on a remote A100.

```bash
# One-time setup:
pip install modal
modal token set
```

**Note:** You'll need a HuggingFace token with access to Meta LLaMA models.
Set it as a Modal secret or pass it as an environment variable.

In [ ]:
# Self-contained Modal script for QLoRA fine-tuning
# Run this cell, then execute: modal run <this_script>
# Or call the function directly via .remote()

MODAL_SCRIPT = '''
import modal

image = (
    modal.Image.debian_slim(python_version="3.11")
    .pip_install(
        "torch", "transformers", "peft", "bitsandbytes",
        "trl", "datasets", "accelerate", "tqdm",
    )
)

app = modal.App("dl101-lora-finetune")

@app.function(gpu="A100", image=image, timeout=3600)
def finetune_llama_qlora(
    model_name: str = "meta-llama/Llama-3.2-1B",
    dataset_name: str = "mlabonne/guanaco-llama2-1k",
    r: int = 16,
    lora_alpha: int = 32,
    epochs: int = 1,
    batch_size: int = 4,
    accum_steps: int = 4,
    lr: float = 2e-4,
    max_seq_length: int = 512,
) -> dict:
    import torch
    from transformers import (
        AutoModelForCausalLM, AutoTokenizer,
        BitsAndBytesConfig, TrainingArguments,
    )
    from peft import LoraConfig, get_peft_model
    from trl import SFTTrainer
    from datasets import load_dataset

    print(f"GPU: {torch.cuda.get_device_name()}")

    # Quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name, quantization_config=bnb_config, device_map="auto",
    )
    model.config.use_cache = False
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    peft_config = LoraConfig(
        r=r, lora_alpha=lora_alpha, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    dataset = load_dataset(dataset_name, split="train")

    training_args = TrainingArguments(
        output_dir="/tmp/lora-output",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=accum_steps,
        learning_rate=lr,
        weight_decay=0.01,
        warmup_ratio=0.03,
        logging_steps=10,
        fp16=True,
        optim="paged_adamw_32bit",
        gradient_checkpointing=True,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model, train_dataset=dataset, args=training_args,
        tokenizer=tokenizer, max_seq_length=max_seq_length,
    )

    trainer.train()

    # Quick inference test
    model.eval()
    prompt = "What are the benefits of LoRA?"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.7, do_sample=True)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return {
        "gpu_name": torch.cuda.get_device_name(),
        "peak_memory_mb": torch.cuda.max_memory_allocated() / 1024**2,
        "train_loss": trainer.state.log_history[-1].get("train_loss", None),
        "sample_response": response,
    }
'''

print("Modal script defined.")
print("To run: save as a .py file and execute 'modal run <file>.py'")
print("Or import and call finetune_llama_qlora.remote() from a notebook with Modal configured.")

## 11. Reusable Implementation

The from-scratch LoRA classes are available in `src/models/lora.py`:

```python
from src.models.lora import LoRALinear, LoRAModel
```

For production fine-tuning, use the PEFT library directly — it handles
quantization integration, adapter I/O, and merging.

In [ ]:
from src.models.lora import LoRALinear as LL, LoRAModel as LM

# Quick smoke test
test_linear = nn.Linear(32, 32)
test_lora = LL(test_linear, r=4, alpha=8)
x = torch.randn(1, 32)
out = test_lora(x)
print(f"LoRALinear output shape: {out.shape}")
print("All src/ imports verified.")

## 12. Key Takeaways

1. **LoRA decomposes weight updates into low-rank matrices.** Instead of updating $d \times d$ parameters, update $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times d}$ where $r \ll d$. Typically 0.1-1% of total parameters.

2. **B=0 initialization preserves the pretrained model.** Since $\Delta W = BA = 0$ at start, training begins from the exact pretrained weights.

3. **QLoRA combines 4-bit quantization with LoRA.** The base model is frozen at 4 bits (~0.5 GB for 1B params), while LoRA adapters train in fp16. This enables fine-tuning a 1B model on a single consumer GPU.

4. **Adapters are tiny and portable.** A LoRA adapter for a 1B model is ~10-50 MB. You can share one base model and swap adapters for different tasks.

5. **Use PEFT + TRL for production.** The from-scratch implementation is educational; PEFT handles quantization, merging, and I/O correctly.

6. **Comparison:**

| | Full Fine-Tuning | LoRA | QLoRA |
|---|---|---|---|
| Trainable params | 100% | 0.1-1% | 0.1-1% |
| Base model memory | fp16 (2B/param) | fp16 (2B/param) | 4-bit (0.5B/param) |
| GPU for 1B model | ~12 GB | ~2.5 GB | ~1 GB |
| Quality | Best | Near-full | Near-full |

### Further Reading

- Hu et al. (2021). *LoRA: Low-Rank Adaptation of Large Language Models.* https://arxiv.org/abs/2106.09685
- Dettmers et al. (2023). *QLoRA: Efficient Finetuning of Quantized LLMs.* https://arxiv.org/abs/2305.14314
- PEFT docs: https://huggingface.co/docs/peft
- TRL docs: https://huggingface.co/docs/trl